**Catalog & Schema Setup**

In [0]:
## Setup Hierarchy

#Define Namesopace
catalog_name = "dev"
schema_name = "ecommerce_governed"

try:
    #Create Catalog 
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
    spark.sql(f"USE CATALOG {catalog_name}")

    #Create Schema 
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
    spark.sql(f"USE SCHEMA {schema_name}")

except Exception as e:
    print(f"Could not create Catalog or Schema.")


**Registering Managed Tables**

In [0]:
#Define source paths
base_path = "/Volumes/workspace/ecommerce/ecommerce_data/delta"

paths = {
  "bronze_events": f"{base_path}/bronze_events",
  "silver_events": f"{base_path}/silver_events",
  "gold_product": f"{base_path}/gold_product"
}

## Register tables

def register_table(table_name , source_path):
  print(f"Registering Table: {table_name}")
  #Read existing Delta Files
  df = spark.read.format("delta").load(source_path)
  #Write as Managed Table
  df.write.format("delta").mode("overwrite").saveAsTable(table_name)

  print(f"Registered Table {table_name} successfully")

#Run Registration for all 3 tables
register_table("bronze_events", paths["bronze_events"])
register_table("silver_events", paths["silver_events"])
register_table("gold_events", paths["gold_product"])



**Access Control**

In [0]:
##Permissions
#Strategy - "Principle of Least Privilege"
# -Bronze : Restricted (Engineers Only)
# -Gold : Accessible to Analysts

query_permissions = """
    GRANT SELECT ON TABLE gold_product TO 'analyst@company.com'
    REVOKE SELECT ON TABLE bronze_events from 'analyst@company.com'

    GRANT ALL PRIVILEGES ON TABLE bronze_events TO 'engineer@company.com'
"""
spark.sql(query_permissions)

display(spark.sql("SHOW GRANT ON TABLE gold_product"))


**Secure Views**

In [0]:
from pyspark.sql.functions import col, sha2, concat_ws, lit

## Create Secure Views

# 1. Load silver data
df_silver = spark.read.table("dev.ecommerce_governed.silver_events")

# 2. Apply masking Logic (SHA-256 HASHING)
df_masked = df_silver.select(
    col("event_time"),
    col("event_type"),
    col("product_id"),
    col("price"),
    col("brand"),
    sha2(col("user_id"),256).alias("hashed_user_id")
)

# 3. Create the view
view_name = "marketing_safe_view"
spark.sql(f"""
          CREATE OR REPLACE VIEW {schema_name}.{view_name}
          AS 
          SELECT event_time,
                 event_type,
                 product_id,
                 price,
                 brand,
                 SHA2(CAST(user_id AS STRING), 256) AS hashed_user_id
          FROM {schema_name}.silver_events
          """)

# 4. Validation
display(spark.read.table(f"{schema_name}.{view_name}").limit(5))
